# IMU IK to ID Comparison (JinWoo_MT_IK)

This notebook runs inference from IMU-derived IK angles using `runs_0507_all/best_model.pt` (with fallback to `runs/0507_ik_id_all/best_model.pt`) and compares model outputs against OpenSim inverse dynamics.

Pipeline used here:
- Angle input: **6 Hz zero-phase low-pass**, 4th order
- Angular velocity input: **15 Hz zero-phase low-pass**, 4th order
- Model output: **6 Hz zero-phase low-pass**, 4th order

Because the MTw IK pickle files do not include timestamps, alignment is done by sample index over the shared trial duration.

In [57]:
import sys
import warnings
import pickle
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, find_peaks
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore", message=".*NumPy.*")

In [58]:
# Paths
PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics')
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/JinWoo_MT_IK/outputs/mtw_ik')
OPENSIM_ROOT = Path('/home/metamobility3/Jinwoo')
TELEMETRY_ROOT = PROJECT_ROOT

# User-requested checkpoint path, with robust fallback to current repo structure
CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / 'runs/0510_ik_id_no_stair_epic_only' / 'best_model.pt'
]
CHECKPOINT = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if CHECKPOINT is None:
    raise FileNotFoundError(f'No checkpoint found in candidates: {CHECKPOINT_CANDIDATES}')

# Filter specs (requested)
ANGLE_CUTOFF_HZ = 6.0
VEL_CUTOFF_HZ = 6.0 # 15.0
OUT_CUTOFF_HZ = 6.0
FILTER_ORDER = 4

# Data assumptions
DEFAULT_FS_HZ = 100.0  # used when timestamps are unavailable (IMU IK pkl)
PREFERRED_EXO_FOLDER = 'awinda'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN, TransformerMoment, GaussianDiffusion1D

print(f'Checkpoint: {CHECKPOINT}')
print(f'Device: {DEVICE}')

Checkpoint: /home/metamobility3/Jinwoo/os_kinetics/runs/0510_ik_id_no_stair_epic_only/best_model.pt
Device: cuda


In [59]:
def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path, 'r') as f:
        header_end = None
        for i, line in enumerate(f):
            if line.strip().lower() == 'endheader':
                header_end = i
                break
    if header_end is None:
        raise RuntimeError(f'endheader not found: {path}')
    df = pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1)
    return df.set_index('time')


def causal_lpf(x: np.ndarray, fs_hz: float, cutoff_hz: float, order: int = 4) -> np.ndarray:
    """Zero-phase low-pass filter (kept function name for compatibility)."""
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1:
        raise ValueError('lpf expects 1D input')
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    try:
        y = sosfiltfilt(sos, x)
    except ValueError:
        # Very short signals can violate filtfilt padding constraints.
        return x.copy()
    return y


def causal_lpf_multichannel(X: np.ndarray, fs_hz: float, cutoff_hz: float, order: int = 4) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2:
        raise ValueError('lpf_multichannel expects (T, C)')
    return np.column_stack([causal_lpf(X[:, c], fs_hz=fs_hz, cutoff_hz=cutoff_hz, order=order) for c in range(X.shape[1])])


def estimate_subject_mass_kg(subject_dir: Path) -> float:
    opensim_dir = subject_dir / 'opensim'
    if not opensim_dir.exists():
        return np.nan
    no_exo = sorted(opensim_dir.glob('*no_exo.osim'))
    if not no_exo:
        return np.nan
    tree = ET.parse(no_exo[0])
    root = tree.getroot()
    masses = []
    for body in root.findall('.//Body'):
        m = body.find('mass')
        if m is not None and m.text:
            try:
                masses.append(float(m.text))
            except ValueError:
                pass
    if not masses:
        for body in root.findall('.//{*}Body'):
            m = body.find('{*}mass')
            if m is not None and m.text:
                try:
                    masses.append(float(m.text))
                except ValueError:
                    pass
    return float(np.sum(masses)) if masses else np.nan


def filename_to_condition(stem: str) -> str:
    speed_token, cond_token = stem.split('_', 1)  # e.g. 0p8mps_lg
    return f'{cond_token.upper()}_{speed_token}'   # -> LG_0p8mps


def find_id_path(subject: str, condition: str) -> tuple[Path, str]:
    subject_dir = OPENSIM_ROOT / subject
    exo_order = [PREFERRED_EXO_FOLDER, 'hip-exo', 'knee-exo']
    for exo in exo_order:
        p = subject_dir / exo / 'id' / f'{condition}_id.sto'
        if p.exists():
            return p, exo
    return None, None


def find_ik_path(subject: str, condition: str, exo_folder: str = None) -> Path:
    subject_dir = OPENSIM_ROOT / subject
    exo_order = [PREFERRED_EXO_FOLDER, 'hip-exo', 'knee-exo']
    if exo_folder in exo_order:
        exo_order = [exo_folder] + [e for e in exo_order if e != exo_folder]

    for exo in exo_order:
        ik_dir = subject_dir / exo / 'ik'
        exact = ik_dir / f'{condition}_ik.mot'
        if exact.exists():
            return exact
        matches = sorted(ik_dir.glob(f'{condition}*_ik.mot')) if ik_dir.exists() else []
        if matches:
            return matches[0]
    return None


def build_mocap_ik_matrix_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan, dtype=np.float64)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def _subject_token(subject: str) -> str:
    return subject.lower().replace('-', '_')


def _condition_tokens(condition: str) -> tuple[str, str]:
    cond_token, speed_token = condition.lower().split('_', 1)  # lg_0p8mps
    return speed_token, cond_token


def telemetry_npz_path(subject: str, condition: str, exo_type: str) -> Path:
    speed_token, cond_token = _condition_tokens(condition)
    stem = f"{_subject_token(subject)}_{exo_type}_{speed_token}_{cond_token}_exo_on.npz"
    p = TELEMETRY_ROOT / stem
    return p if p.exists() else None


def load_telemetry_torque_lr(npz_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    d = np.load(npz_path, allow_pickle=True)
    t = np.asarray(d['time'], dtype=np.float64)

    # Prefer applied torque channels when available; otherwise fall back to commanded torque.
    if 'applied_R' in d and 'applied_L' in d:
        tr = np.asarray(d['applied_R'], dtype=np.float64)
        tl = np.asarray(d['applied_L'], dtype=np.float64)
    elif 'cmd_R' in d and 'cmd_L' in d:
        tr = np.asarray(d['cmd_R'], dtype=np.float64)
        tl = np.asarray(d['cmd_L'], dtype=np.float64)
    else:
        raise KeyError(f'No applied/cmd torque channels found in {npz_path.name}')

    return t, tr, tl


def interp_telemetry_to_sync_time(
    t_sync: np.ndarray,
    t_telemetry: np.ndarray,
    torque_r: np.ndarray,
    torque_l: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    t_sync_rel = np.asarray(t_sync, dtype=np.float64) - float(t_sync[0])
    t_tel_rel = np.asarray(t_telemetry, dtype=np.float64) - float(t_telemetry[0])

    tor_r = np.interp(t_sync_rel, t_tel_rel, torque_r, left=np.nan, right=np.nan)
    tor_l = np.interp(t_sync_rel, t_tel_rel, torque_l, left=np.nan, right=np.nan)

    # Outside telemetry support window -> assume no applied exo torque.
    tor_r = np.where(np.isfinite(tor_r), tor_r, 0.0)
    tor_l = np.where(np.isfinite(tor_l), tor_l, 0.0)
    return tor_r.astype(np.float64), tor_l.astype(np.float64)


def _first_peak_index_above_threshold(
    sig: np.ndarray,
    fs_hz: float,
    threshold_rad: float,
    start_offset_s: float = 0.25,
) -> tuple[int, str]:
    x = np.asarray(sig, dtype=np.float64).copy()
    if x.ndim != 1:
        raise ValueError('_first_peak_index_above_threshold expects 1D signal')

    valid = np.isfinite(x)
    if valid.sum() < 20:
        raise RuntimeError('Signal has too few finite samples for peak detection')

    # Fill sparse NaNs to keep peak finding stable.
    fill_val = float(np.nanmedian(x[valid]))
    x[~valid] = fill_val

    start_idx = min(len(x) - 1, max(0, int(round(start_offset_s * fs_hz))))
    if start_idx >= len(x) - 5:
        start_idx = 0

    span = float(np.nanmax(x) - np.nanmin(x))
    min_prom = max(np.deg2rad(3.0), 0.10 * span)
    min_dist = max(1, int(round(0.3 * fs_hz)))

    peaks, _ = find_peaks(x, prominence=min_prom, distance=min_dist)
    peaks = peaks[(peaks >= start_idx) & (x[peaks] >= threshold_rad)]

    if len(peaks) > 0:
        return int(peaks[0]), 'peaks_above_threshold'

    # Fallback: first threshold crossing after start, if it exists.
    crossing = np.where(x[start_idx:] >= threshold_rad)[0]
    if len(crossing) > 0:
        return int(start_idx + crossing[0]), 'threshold_crossing_fallback'

    raise RuntimeError('No right hip flexion peak above threshold found')


def estimate_lag_from_first_right_hip_peak_above_threshold(
    imu_pos: np.ndarray,
    mocap_pos: np.ndarray,
    fs_hz: float,
    max_lag_samples: int,
    threshold_deg: float = 15.0,
) -> tuple[int, dict]:
    hip_name = 'hip_flexion_r'
    if hip_name not in IK_DOF_NAMES:
        raise RuntimeError('Right hip flexion channel not found in IK_DOF_NAMES')
    hip_idx = IK_DOF_NAMES.index(hip_name)

    threshold_rad = np.deg2rad(float(threshold_deg))
    imu_idx, imu_method = _first_peak_index_above_threshold(
        imu_pos[:, hip_idx], fs_hz=fs_hz, threshold_rad=threshold_rad
    )
    mocap_idx, mocap_method = _first_peak_index_above_threshold(
        mocap_pos[:, hip_idx], fs_hz=fs_hz, threshold_rad=threshold_rad
    )

    lag_samples = int(imu_idx - mocap_idx)
    clipped = False
    if lag_samples > max_lag_samples:
        lag_samples = max_lag_samples
        clipped = True
    elif lag_samples < -max_lag_samples:
        lag_samples = -max_lag_samples
        clipped = True

    info = {
        'hip_channel': IK_DOF_NAMES[hip_idx],
        'peak_threshold_deg': float(threshold_deg),
        'peak_threshold_rad': float(threshold_rad),
        'imu_peak_idx': int(imu_idx),
        'mocap_peak_idx': int(mocap_idx),
        'imu_peak_method': imu_method,
        'mocap_peak_method': mocap_method,
        'lag_clipped': bool(clipped),
    }
    return lag_samples, info


print('Helpers defined.')

Helpers defined.


In [60]:
# Load checkpoint model + metadata
ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
cfg = ckpt['model_config']
model_type = cfg.get('model_type', 'tcn')

if model_type == 'transformer':
    model = TransformerMoment(
        n_input_channels=cfg['n_input_channels'],
        n_output_channels=cfg['n_output_channels'],
        d_model=cfg['d_model'],
        n_heads=cfg['n_heads'],
        n_layers=cfg['n_layers'],
        d_ff=cfg['d_ff'],
        dropout=cfg['dropout'],
    )
elif model_type == 'diffusion':
    model = GaussianDiffusion1D(
        n_input_channels=cfg['n_input_channels'],
        n_output_channels=cfg['n_output_channels'],
        d_model=cfg['d_model'],
        n_heads=cfg['n_heads'],
        n_layers=cfg['n_layers'],
        d_ff=cfg['d_ff'],
        dropout=cfg['dropout'],
        n_timesteps=cfg.get('n_timesteps', 1000),
        schedule=cfg.get('schedule', 'cosine'),
        predict_epsilon=cfg.get('predict_epsilon', True),
        n_inference_steps=cfg.get('n_inference_steps', 50),
    )
else:
    model = TCN(
        n_input_channels=cfg['n_input_channels'],
        n_output_channels=cfg['n_output_channels'],
        hidden_channels=cfg['hidden_channels'],
        n_blocks=cfg['n_blocks'],
        kernel_size=cfg['kernel_size'],
        dropout=cfg['dropout'],
    )

model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE)
model.eval()

WINDOW_SIZE = int(ckpt.get('window_size', 100))
INPUT_INDICES = list(ckpt.get('input_indices', [6, 9, 10, 13, 16, 17]))
MOMENT_INDICES = list(ckpt.get('moment_indices', [3, 12, 14, 6, 13, 15]))
DOF_NAMES = list(ckpt.get('dof_names', ['hip_flexion', 'knee_angle', 'ankle_angle']))

# Right/left index split for unilateral-paired inference
H = len(INPUT_INDICES) // 2
INPUT_IDX_R = INPUT_INDICES[:H]
INPUT_IDX_L = INPUT_INDICES[H:]

ID_COLS = [
    'hip_flexion_r_moment',
    'knee_angle_r_moment',
    'ankle_angle_r_moment',
    'hip_flexion_l_moment',
    'knee_angle_l_moment',
    'ankle_angle_l_moment',
]

print('Model config:', cfg)
print('window_size:', WINDOW_SIZE)
print('input_indices:', INPUT_INDICES)
print('moment_indices:', MOMENT_INDICES)
print('DOF names:', DOF_NAMES)

Model config: {'model_type': 'tcn', 'n_input_channels': 6, 'n_output_channels': 3, 'hidden_channels': 80, 'n_blocks': 5, 'kernel_size': 5, 'dropout': 0.1}
window_size: 100
input_indices: [6, 9, 10, 13, 16, 17]
moment_indices: [3, 12, 14, 6, 13, 15]
DOF names: ['hip_flexion', 'knee_angle', 'ankle_angle']


In [61]:
@torch.no_grad()
def causal_infer_one_side(pos_3: np.ndarray, vel_3: np.ndarray) -> np.ndarray:
    """
    pos_3, vel_3: (T, 3), already filtered, radians / rad/s.
    Returns: (T, 3) model output in N·m/kg.
    """
    x = np.concatenate([pos_3, vel_3], axis=1).astype(np.float32)  # (T, 6)
    n = x.shape[0]
    W = WINDOW_SIZE
    c_out = cfg['n_output_channels']

    if n < W:
        raise ValueError(f'Trial too short: n={n}, window={W}')

    pred = np.zeros((n, c_out), dtype=np.float64)

    def _fwd(start: int) -> np.ndarray:
        xw = np.ascontiguousarray(x[start:start + W].T)
        xt = torch.from_numpy(xw).unsqueeze(0).to(DEVICE)
        yw = model(xt).squeeze(0).detach().cpu().numpy().T  # (W, c_out)
        return yw

    # Bootstrap first window
    y0 = _fwd(0)
    pred[:W] = y0

    # Causal rolling: keep newest sample from each subsequent window
    for start in range(1, n - W + 1):
        yw = _fwd(start)
        pred[start + W - 1] = yw[W - 1]

    return pred.astype(np.float32)


def run_bilateral_inference(pos_full: np.ndarray, vel_full: np.ndarray) -> np.ndarray:
    pred_r = causal_infer_one_side(pos_full[:, INPUT_IDX_R], vel_full[:, INPUT_IDX_R])
    pred_l = causal_infer_one_side(pos_full[:, INPUT_IDX_L], vel_full[:, INPUT_IDX_L])
    return np.concatenate([pred_r, pred_l], axis=1)


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    """Build full IK vector (T, len(IK_DOF_NAMES)) from MTw IK pickle channels (degrees)."""
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)

    key_map = {
        'hip_flexion_r': 'hip_flexion_r',
        'knee_angle_r': 'knee_flexion_r',
        'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l',
        'knee_angle_l': 'knee_flexion_l',
        'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {
        'knee_angle_r': -1.0,
        'knee_angle_l': -1.0,
    }

    for ik_name, pkl_name in key_map.items():
        if pkl_name not in imu_dict:
            raise KeyError(f'Missing channel in pkl: {pkl_name}')
        idx = IK_DOF_NAMES.index(ik_name)
        sign = sign_map.get(ik_name, 1.0)
        pos_deg[:, idx] = sign * np.asarray(imu_dict[pkl_name], dtype=np.float64)

    return pos_deg

In [62]:
# Discover candidate trials
records = []
for subj_dir in sorted(IMU_IK_ROOT.glob('AB*')):
    if not subj_dir.is_dir():
        continue
    subject = subj_dir.name
    subj_opensim_dir = OPENSIM_ROOT / subject
    mass_kg = estimate_subject_mass_kg(subj_opensim_dir)

    for pkl_path in sorted(subj_dir.glob('*.pkl')):
        cond = filename_to_condition(pkl_path.stem)
        id_path, exo_folder = find_id_path(subject, cond)
        records.append({
            'subject': subject,
            'pkl_file': pkl_path.name,
            'condition': cond,
            'id_found': id_path is not None,
            'id_path': id_path,
            'exo_folder': exo_folder,
            'mass_kg': mass_kg,
        })

manifest_df = pd.DataFrame(records)
display(manifest_df)

print('Total IMU IK files:', len(manifest_df))
print('Matched with ID:', int(manifest_df['id_found'].sum()))
print('Unmatched (will skip):', int((~manifest_df['id_found']).sum()))

,subject,pkl_file,condition,id_found,id_path,exo_folder,mass_kg
0,AB01_Jinwoo,0p8mps_lg.pkl,LG_0p8mps,True,/home/metamobility3/Jinwoo/AB01_Jinwoo/awinda/...,awinda,88.0
1,AB01_Jinwoo,0p8mps_ra.pkl,RA_0p8mps,True,/home/metamobility3/Jinwoo/AB01_Jinwoo/awinda/...,awinda,88.0
2,AB01_Jinwoo,0p8mps_rd.pkl,RD_0p8mps,True,/home/metamobility3/Jinwoo/AB01_Jinwoo/awinda/...,awinda,88.0
3,AB01_Jinwoo,1p2mps_lg.pkl,LG_1p2mps,True,/home/metamobility3/Jinwoo/AB01_Jinwoo/awinda/...,awinda,88.0
4,AB01_Jinwoo,1p6mps_lg.pkl,LG_1p6mps,True,/home/metamobility3/Jinwoo/AB01_Jinwoo/awinda/...,awinda,88.0
5,AB02_Oscar,0p8mps_lg.pkl,LG_0p8mps,True,/home/metamobility3/Jinwoo/AB02_Oscar/awinda/i...,awinda,71.1
6,AB02_Oscar,0p8mps_ra.pkl,RA_0p8mps,True,/home/metamobility3/Jinwoo/AB02_Oscar/awinda/i...,awinda,71.1
7,AB02_Oscar,0p8mps_rd.pkl,RD_0p8mps,True,/home/metamobility3/Jinwoo/AB02_Oscar/awinda/i...,awinda,71.1
8,AB02_Oscar,1p2mps_lg.pkl,LG_1p2mps,True,/home/metamobility3/Jinwoo/AB02_Oscar/awinda/i...,awinda,71.1
9,AB02_Oscar,1p6mps_lg.pkl,LG_1p6mps,True,/home/metamobility3/Jinwoo/AB02_Oscar/awinda/i...,awinda,71.1


Total IMU IK files: 20
Matched with ID: 20
Unmatched (will skip): 0


In [63]:
# Run inference + comparison
TRIAL_DATA = {}
rows = []

IK_ALIGN_MAX_LAG_SEC = 30.0

matched_df = manifest_df[manifest_df['id_found']].reset_index(drop=True)
total_trials = len(matched_df)
print(f'Running inference for {total_trials} matched trials...')

for i, row in matched_df.iterrows():
    trial_idx = i + 1
    subject = row['subject']
    pkl_file = row['pkl_file']
    cond = row['condition']
    exo_folder = row['exo_folder']
    id_path = Path(row['id_path'])
    mass_kg = float(row['mass_kg']) if pd.notna(row['mass_kg']) else np.nan
    trial_key = f'{subject}::{cond}'

    print(f'[{trial_idx:02d}/{total_trials:02d}] START {trial_key} ({pkl_file})', flush=True)

    try:
        pkl_path = IMU_IK_ROOT / subject / pkl_file
        imu = pickle.load(open(pkl_path, 'rb'))
        pos_deg = build_model_input_from_pkl(imu)              # (T_imu, 23)
        pos_rad = np.deg2rad(pos_deg)

        mocap_ik_path = find_ik_path(subject, cond, exo_folder=exo_folder)
        if mocap_ik_path is None:
            raise FileNotFoundError(f'Mocap IK not found for {trial_key}')

        # OpenSim timelines
        id_df = parse_opensim_table(id_path)
        ik_df = parse_opensim_table(mocap_ik_path)

        t_id = id_df.index.to_numpy(dtype=np.float64)
        fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else DEFAULT_FS_HZ

        mocap_pos_rad = build_mocap_ik_matrix_rad(ik_df)

        # Estimate lag from first RIGHT hip-flexion peak above 15 deg.
        # Positive lag means IMU starts later than mocap and is shifted forward.
        imu_pos_for_align = causal_lpf_multichannel(pos_rad, fs_hz=fs_hz, cutoff_hz=ANGLE_CUTOFF_HZ, order=FILTER_ORDER)
        mocap_pos_for_align = causal_lpf_multichannel(mocap_pos_rad, fs_hz=fs_hz, cutoff_hz=ANGLE_CUTOFF_HZ, order=FILTER_ORDER)
        max_lag_samples = int(round(IK_ALIGN_MAX_LAG_SEC * fs_hz))
        lag_samples, peak_info = estimate_lag_from_first_right_hip_peak_above_threshold(
            imu_pos_for_align,
            mocap_pos_for_align,
            fs_hz=fs_hz,
            max_lag_samples=max_lag_samples,
            threshold_deg=15.0,
        )

        start_imu = max(lag_samples, 0)
        start_ref = max(-lag_samples, 0)
        n_sync = min(
            len(pos_rad) - start_imu,
            len(mocap_pos_rad) - start_ref,
            len(t_id) - start_ref,
        )
        if n_sync < WINDOW_SIZE:
            raise RuntimeError(f'Not enough synced samples after lag alignment: {n_sync}')

        pos_rad_sync = pos_rad[start_imu:start_imu + n_sync]
        mocap_pos_rad_sync = mocap_pos_rad[start_ref:start_ref + n_sync]
        t_sync = t_id[start_ref:start_ref + n_sync]

        # Build OpenSim ID matrix (N·m), align to mocap timeline, and apply requested 6 Hz zero-phase LPF.
        id_nm = np.zeros((len(t_id), len(ID_COLS)), dtype=np.float64)
        for c, col in enumerate(ID_COLS):
            id_nm[:, c] = id_df[col].to_numpy(dtype=np.float64) if col in id_df.columns else np.nan
        id_nm_sync = id_nm[start_ref:start_ref + n_sync]

        # Subtract exoskeleton applied torque from Mocap ID before comparison.
        # Hip telemetry drives hip moments; knee telemetry drives knee moments.
        exo_torque_sync = np.zeros_like(id_nm_sync, dtype=np.float64)

        hip_npz_path = telemetry_npz_path(subject, cond, exo_type='hip')
        if hip_npz_path is not None:
            t_tel, tor_r, tor_l = load_telemetry_torque_lr(hip_npz_path)
            tor_r_sync, tor_l_sync = interp_telemetry_to_sync_time(t_sync, t_tel, tor_r, tor_l)
            exo_torque_sync[:, 0] += tor_r_sync  # hip_flexion_r_moment
            exo_torque_sync[:, 3] += tor_l_sync  # hip_flexion_l_moment

        knee_npz_path = telemetry_npz_path(subject, cond, exo_type='knee')
        if knee_npz_path is not None:
            t_tel, tor_r, tor_l = load_telemetry_torque_lr(knee_npz_path)
            tor_r_sync, tor_l_sync = interp_telemetry_to_sync_time(t_sync, t_tel, tor_r, tor_l)
            exo_torque_sync[:, 1] += tor_r_sync  # knee_angle_r_moment
            exo_torque_sync[:, 4] += tor_l_sync  # knee_angle_l_moment

        id_nm_net_sync = id_nm_sync - exo_torque_sync

        # Requested filters: 6 Hz on angle, 15 Hz on velocity, 6 Hz on model output, and 6 Hz on OpenSim ID (all zero-phase, order 4)
        pos_f = causal_lpf_multichannel(pos_rad_sync, fs_hz=fs_hz, cutoff_hz=ANGLE_CUTOFF_HZ, order=FILTER_ORDER)
        vel_raw = np.gradient(pos_f, 1.0 / fs_hz, axis=0)
        vel_f = causal_lpf_multichannel(vel_raw, fs_hz=fs_hz, cutoff_hz=VEL_CUTOFF_HZ, order=FILTER_ORDER)

        pred_nmpkg = run_bilateral_inference(pos_f, vel_f)
        pred_nmpkg_f = causal_lpf_multichannel(pred_nmpkg, fs_hz=fs_hz, cutoff_hz=OUT_CUTOFF_HZ, order=FILTER_ORDER)
        id_nm_f = causal_lpf_multichannel(id_nm_net_sync, fs_hz=fs_hz, cutoff_hz=OUT_CUTOFF_HZ, order=FILTER_ORDER)

        # Convert to N·m when mass is available
        pred_nm_f = pred_nmpkg_f * mass_kg if np.isfinite(mass_kg) else np.full_like(pred_nmpkg_f, np.nan)

        # Shared aligned window
        n_common = min(len(pred_nmpkg_f), len(id_nm_f))
        pred_nmpkg_c = pred_nmpkg_f[:n_common]
        pred_nm_c = pred_nm_f[:n_common]
        id_nm_c = id_nm_f[:n_common]
        t_c = t_sync[:n_common]

        if np.isfinite(mass_kg):
            id_nmpkg_c = id_nm_c / mass_kg
        else:
            id_nmpkg_c = np.full_like(id_nm_c, np.nan)

        # Metrics in N·m/kg (primary) and N·m (if mass available)
        metrics = []
        for c in range(len(ID_COLS)):
            m = {}
            v_kg = np.isfinite(pred_nmpkg_c[:, c]) & np.isfinite(id_nmpkg_c[:, c])
            if v_kg.sum() > 1:
                e_kg = pred_nmpkg_c[v_kg, c] - id_nmpkg_c[v_kg, c]
                corr_kg = np.corrcoef(pred_nmpkg_c[v_kg, c], id_nmpkg_c[v_kg, c])[0, 1]
                m['rmse_nmpkg'] = float(np.sqrt(np.mean(e_kg ** 2)))
                m['r2_nmpkg'] = float(corr_kg ** 2)
            else:
                m['rmse_nmpkg'] = np.nan
                m['r2_nmpkg'] = np.nan

            v_nm = np.isfinite(pred_nm_c[:, c]) & np.isfinite(id_nm_c[:, c])
            if v_nm.sum() > 1:
                e_nm = pred_nm_c[v_nm, c] - id_nm_c[v_nm, c]
                corr_nm = np.corrcoef(pred_nm_c[v_nm, c], id_nm_c[v_nm, c])[0, 1]
                m['rmse_nm'] = float(np.sqrt(np.mean(e_nm ** 2)))
                m['r2_nm'] = float(corr_nm ** 2)
            else:
                m['rmse_nm'] = np.nan
                m['r2_nm'] = np.nan

            metrics.append(m)

        TRIAL_DATA[trial_key] = {
            'subject': subject,
            'condition': cond,
            'pkl_file': pkl_file,
            'id_path': str(id_path),
            'mocap_ik_path': str(mocap_ik_path),
            'hip_telemetry_npz': str(hip_npz_path) if hip_npz_path is not None else None,
            'knee_telemetry_npz': str(knee_npz_path) if knee_npz_path is not None else None,
            'mass_kg': mass_kg,
            'lag_samples': int(lag_samples),
            'lag_seconds': float(lag_samples / fs_hz),
            'align_method': 'first_right_hip_peak_gt15deg',
            'hip_channel': peak_info['hip_channel'],
            'peak_threshold_deg': float(peak_info['peak_threshold_deg']),
            'imu_peak_idx': int(peak_info['imu_peak_idx']),
            'mocap_peak_idx': int(peak_info['mocap_peak_idx']),
            'imu_peak_method': peak_info['imu_peak_method'],
            'mocap_peak_method': peak_info['mocap_peak_method'],
            'lag_clipped': bool(peak_info['lag_clipped']),
            't': t_c,
            'imu_ik_rad': pos_rad_sync[:n_common],
            'mocap_ik_rad': mocap_pos_rad_sync[:n_common],
            'pred_nmpkg': pred_nmpkg_c,
            'pred_nm': pred_nm_c,
            'id_nmpkg': id_nmpkg_c,
            'id_nm': id_nm_c,
            'metrics': metrics,
        }

        rows.append({
            'trial': trial_key,
            'subject': subject,
            'condition': cond,
            'pkl_file': pkl_file,
            'mass_kg': mass_kg,
            'hip_torque_subtracted': bool(hip_npz_path is not None),
            'knee_torque_subtracted': bool(knee_npz_path is not None),
            'align_method': 'first_right_hip_peak_gt15deg',
            'hip_channel': peak_info['hip_channel'],
            'lag_samples': int(lag_samples),
            'lag_seconds': float(lag_samples / fs_hz),
            'peak_threshold_deg': float(peak_info['peak_threshold_deg']),
            'imu_peak_idx': int(peak_info['imu_peak_idx']),
            'mocap_peak_idx': int(peak_info['mocap_peak_idx']),
            'lag_clipped': bool(peak_info['lag_clipped']),
            'n_common': n_common,
            'mean_rmse_nmpkg': float(np.nanmean([m['rmse_nmpkg'] for m in metrics])),
            'mean_r2_nmpkg': float(np.nanmean([m['r2_nmpkg'] for m in metrics])),
            'mean_rmse_nm': float(np.nanmean([m['rmse_nm'] for m in metrics])),
            'mean_r2_nm': float(np.nanmean([m['r2_nm'] for m in metrics])),
        })

        print(
            f'[{trial_idx:02d}/{total_trials:02d}] DONE  {trial_key} '
            f"| lag={lag_samples:+d} samples ({lag_samples / fs_hz:+.3f} s), "
            f"hip={peak_info['hip_channel']} first peak>{peak_info['peak_threshold_deg']:.1f}deg "
            f"(imu={peak_info['imu_peak_idx']}, mocap={peak_info['mocap_peak_idx']}), "
            f"torque_subtract[hip={hip_npz_path is not None}, knee={knee_npz_path is not None}], "
            f'n_common={n_common}',
            flush=True,
        )
    except Exception as e:
        print(f'[{trial_idx:02d}/{total_trials:02d}] FAIL  {trial_key} | {type(e).__name__}: {e}', flush=True)

summary_df = pd.DataFrame(rows).sort_values(['subject', 'condition']).reset_index(drop=True)
display(summary_df)
print('Computed trials:', len(summary_df))

Running inference for 20 matched trials...
[01/20] START AB01_Jinwoo::LG_0p8mps (0p8mps_lg.pkl)
[01/20] FAIL  AB01_Jinwoo::LG_0p8mps | RuntimeError: No right hip flexion peak above threshold found
[02/20] START AB01_Jinwoo::RA_0p8mps (0p8mps_ra.pkl)
[02/20] DONE  AB01_Jinwoo::RA_0p8mps | lag=+778 samples (+7.780 s), hip=hip_flexion_r first peak>15.0deg (imu=1045, mocap=267), torque_subtract[hip=True, knee=True], n_common=2569
[03/20] START AB01_Jinwoo::RD_0p8mps (0p8mps_rd.pkl)
[03/20] DONE  AB01_Jinwoo::RD_0p8mps | lag=+1585 samples (+15.850 s), hip=hip_flexion_r first peak>15.0deg (imu=1822, mocap=237), torque_subtract[hip=False, knee=True], n_common=1866
[04/20] START AB01_Jinwoo::LG_1p2mps (1p2mps_lg.pkl)
[04/20] FAIL  AB01_Jinwoo::LG_1p2mps | RuntimeError: No right hip flexion peak above threshold found
[05/20] START AB01_Jinwoo::LG_1p6mps (1p6mps_lg.pkl)
[05/20] FAIL  AB01_Jinwoo::LG_1p6mps | RuntimeError: No right hip flexion peak above threshold found
[06/20] START AB02_Oscar::

,trial,subject,condition,pkl_file,mass_kg,hip_torque_subtracted,knee_torque_subtracted,align_method,hip_channel,lag_samples,lag_seconds,peak_threshold_deg,imu_peak_idx,mocap_peak_idx,lag_clipped,n_common,mean_rmse_nmpkg,mean_r2_nmpkg,mean_rmse_nm,mean_r2_nm
0,AB01_Jinwoo::RA_0p8mps,AB01_Jinwoo,RA_0p8mps,0p8mps_ra.pkl,88.0,True,True,first_right_hip_peak_gt15deg,hip_flexion_r,778,7.78,15.0,1045,267,False,2569,0.536047,0.001014,47.172155,0.001014
1,AB01_Jinwoo::RD_0p8mps,AB01_Jinwoo,RD_0p8mps,0p8mps_rd.pkl,88.0,False,True,first_right_hip_peak_gt15deg,hip_flexion_r,1585,15.85,15.0,1822,237,False,1866,0.412993,0.000874,36.343379,0.000874
2,AB02_Oscar::LG_0p8mps,AB02_Oscar,LG_0p8mps,0p8mps_lg.pkl,71.1,True,False,first_right_hip_peak_gt15deg,hip_flexion_r,1243,12.43,15.0,1327,84,False,8000,0.133388,0.694819,9.483912,0.694819
3,AB02_Oscar::LG_1p2mps,AB02_Oscar,LG_1p2mps,1p2mps_lg.pkl,71.1,True,False,first_right_hip_peak_gt15deg,hip_flexion_r,291,2.91,15.0,382,91,False,8000,0.158736,0.842716,11.286117,0.842716
4,AB02_Oscar::LG_1p6mps,AB02_Oscar,LG_1p6mps,1p6mps_lg.pkl,71.1,True,False,first_right_hip_peak_gt15deg,hip_flexion_r,263,2.63,15.0,330,67,False,8000,0.232057,0.772319,16.499240,0.772319
5,AB02_Oscar::RA_0p8mps,AB02_Oscar,RA_0p8mps,0p8mps_ra.pkl,71.1,True,True,first_right_hip_peak_gt15deg,hip_flexion_r,292,2.92,15.0,404,112,False,8000,0.192967,0.866854,13.719983,0.866854
6,AB02_Oscar::RD_0p8mps,AB02_Oscar,RD_0p8mps,0p8mps_rd.pkl,71.1,False,True,first_right_hip_peak_gt15deg,hip_flexion_r,760,7.60,15.0,812,52,False,8000,0.194784,0.768581,13.849123,0.768581
7,AB03_Ilseung::LG_0p8mps,AB03_Ilseung,LG_0p8mps,0p8mps_lg.pkl,84.4,True,False,first_right_hip_peak_gt15deg,hip_flexion_r,232,2.32,15.0,377,145,False,6850,0.116260,0.812294,9.812319,0.812294
8,AB03_Ilseung::LG_1p2mps,AB03_Ilseung,LG_1p2mps,1p2mps_lg.pkl,84.4,True,False,first_right_hip_peak_gt15deg,hip_flexion_r,269,2.69,15.0,371,102,False,8000,0.120956,0.898728,10.208719,0.898728
9,AB03_Ilseung::LG_1p6mps,AB03_Ilseung,LG_1p6mps,1p6mps_lg.pkl,84.4,True,False,first_right_hip_peak_gt15deg,hip_flexion_r,228,2.28,15.0,350,122,False,8000,0.152022,0.912571,12.830631,0.912571


Computed trials: 17


In [64]:
# Channel-wise summary table
CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]

detail_rows = []
for trial, d in TRIAL_DATA.items():
    for c, ch in enumerate(CHANNELS):
        m = d['metrics'][c]
        detail_rows.append({
            'trial': trial,
            'channel': ch,
            'rmse_nmpkg': m['rmse_nmpkg'],
            'r2_nmpkg': m['r2_nmpkg'],
            'rmse_nm': m['rmse_nm'],
            'r2_nm': m['r2_nm'],
        })

detail_df = pd.DataFrame(detail_rows)
display(detail_df)

print('Mean by channel (N·m/kg):')
display(detail_df.groupby('channel')[['rmse_nmpkg', 'r2_nmpkg']].mean().sort_index())

,trial,channel,rmse_nmpkg,r2_nmpkg,rmse_nm,r2_nm
0,AB01_Jinwoo::RA_0p8mps,hip_flexion_r,0.350908,7.523862e-07,30.879939,7.523862e-07
1,AB01_Jinwoo::RA_0p8mps,knee_angle_r,0.793553,2.015976e-03,69.832702,2.015976e-03
2,AB01_Jinwoo::RA_0p8mps,ankle_angle_r,0.667999,1.183805e-03,58.783926,1.183805e-03
3,AB01_Jinwoo::RA_0p8mps,hip_flexion_l,0.373332,2.727500e-04,32.853252,2.727500e-04
4,AB01_Jinwoo::RA_0p8mps,knee_angle_l,0.379242,9.603013e-04,33.373299,9.603013e-04
...,...,...,...,...,...,...
97,AB08_Seokhyun::LG_1p6mps,knee_angle_r,0.776931,1.591191e-04,55.861328,1.591191e-04
98,AB08_Seokhyun::LG_1p6mps,ankle_angle_r,0.851830,1.414147e-04,61.246578,1.414147e-04
99,AB08_Seokhyun::LG_1p6mps,hip_flexion_l,1.236461,9.006750e-05,88.901519,9.006750e-05
100,AB08_Seokhyun::LG_1p6mps,knee_angle_l,0.731088,7.467294e-06,52.565218,7.467294e-06


Mean by channel (N·m/kg):


,rmse_nmpkg,r2_nmpkg
channel,,
ankle_angle_l,0.381807,0.544044
ankle_angle_r,0.395496,0.526223
hip_flexion_l,0.383708,0.504803
hip_flexion_r,0.403852,0.485763
knee_angle_l,0.313498,0.473107
knee_angle_r,0.370663,0.423559


In [65]:
# Interactive trial plot
if not TRIAL_DATA:
    raise RuntimeError('No trials available. Check path mapping and ID availability.')

trial_keys = sorted(TRIAL_DATA.keys())
trial_dd = widgets.Dropdown(options=trial_keys, value=trial_keys[0], description='Trial:')
unit_dd = widgets.Dropdown(options=['N·m/kg', 'N·m'], value='N·m/kg', description='Unit:')
time_range = widgets.FloatRangeSlider(
    description='Time (s):',
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='700px')
)
out = widgets.Output()


def set_time_slider_for_trial(trial_key: str):
    t = TRIAL_DATA[trial_key]['t']
    t_rel = t - t[0]
    t_min = float(t_rel[0])
    t_max = float(t_rel[-1])
    span = max(t_max - t_min, 1e-6)

    time_range.min = t_min
    time_range.max = t_max
    time_range.step = max(span / 500.0, 1e-3)
    time_range.value = (t_min, t_max)


def draw_trial(trial_key: str, unit: str, t_window: tuple[float, float]):
    d = TRIAL_DATA[trial_key]
    t = d['t'] - d['t'][0]

    if unit == 'N·m/kg':
        y_pred = d['pred_nmpkg']
        y_id = d['id_nmpkg']
        rmse_key = 'rmse_nmpkg'
        r2_key = 'r2_nmpkg'
        ylab = 'N·m/kg'
    else:
        y_pred = d['pred_nm']
        y_id = d['id_nm']
        rmse_key = 'rmse_nm'
        r2_key = 'r2_nm'
        ylab = 'N·m'

    t0, t1 = t_window
    names = ['Hip R', 'Knee R', 'Ankle R', 'Hip L', 'Knee L', 'Ankle L']
    fig, axs = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    axs = axs.reshape(-1)

    for c in range(6):
        ax = axs[c]
        ax.plot(t, y_id[:, c], label='OpenSim ID', color='#1e88e5', lw=1.8)
        ax.plot(t, y_pred[:, c], label='Model', color='#e53935', lw=1.4, ls='--')
        m = d['metrics'][c]
        ax.set_title(f"{names[c]} | RMSE={m[rmse_key]:.3f}, R²={m[r2_key]:.3f}")
        ax.axhline(0.0, color='gray', lw=0.7, ls=':')
        ax.set_xlim(t0, t1)
        ax.set_ylabel(ylab)
        ax.spines[['top', 'right']].set_visible(False)
        if c == 0:
            ax.legend(loc='upper right')

    axs[-2].set_xlabel('Time (s)')
    axs[-1].set_xlabel('Time (s)')
    fig.suptitle(
        f"{trial_key} | mass={d['mass_kg']:.2f} kg | lag={d.get('lag_samples', 0):+d} samples ({d.get('lag_seconds', 0.0):+.3f} s) | filters: 6/15/6 Hz zero-phase (order 4)",
        fontsize=12,
    )
    fig.tight_layout()
    plt.show()


def redraw(*_):
    out.clear_output(wait=True)
    with out:
        draw_trial(trial_dd.value, unit_dd.value, time_range.value)


def on_trial_change(change):
    if change.get('name') == 'value':
        set_time_slider_for_trial(change['new'])
        redraw()


trial_dd.observe(on_trial_change, names='value')
unit_dd.observe(redraw, names='value')
time_range.observe(redraw, names='value')

set_time_slider_for_trial(trial_dd.value)
display(widgets.VBox([widgets.HBox([trial_dd, unit_dd]), time_range]), out)
redraw()

Output()

In [66]:
# Interactive IMU IK vs Mocap IK (after sync)
if not TRIAL_DATA:
    raise RuntimeError('No trials available. Run inference cell first.')

IK_PLOT_CHANNELS = {
    'Right': [('Hip Flexion', 'hip_flexion_r'), ('Knee Angle', 'knee_angle_r'), ('Ankle Angle', 'ankle_angle_r')],
    'Left': [('Hip Flexion', 'hip_flexion_l'), ('Knee Angle', 'knee_angle_l'), ('Ankle Angle', 'ankle_angle_l')],
}

ik_trial_keys = sorted(TRIAL_DATA.keys())
ik_trial_dd = widgets.Dropdown(options=ik_trial_keys, value=ik_trial_keys[0], description='Trial:')
ik_side_dd = widgets.Dropdown(options=['Right', 'Left'], value='Right', description='Side:')
ik_unit_dd = widgets.Dropdown(options=['deg', 'rad'], value='deg', description='Unit:')
ik_time_range = widgets.FloatRangeSlider(
    description='Time (s):',
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='700px')
)
ik_out = widgets.Output()


def set_ik_time_slider_for_trial(trial_key: str):
    t = TRIAL_DATA[trial_key]['t']
    t_rel = t - t[0]
    t_min = float(t_rel[0])
    t_max = float(t_rel[-1])
    span = max(t_max - t_min, 1e-6)

    ik_time_range.min = t_min
    ik_time_range.max = t_max
    ik_time_range.step = max(span / 500.0, 1e-3)
    ik_time_range.value = (t_min, t_max)


def compute_rmse_r2(y_pred: np.ndarray, y_ref: np.ndarray) -> tuple[float, float]:
    valid = np.isfinite(y_pred) & np.isfinite(y_ref)
    if valid.sum() < 2:
        return np.nan, np.nan

    e = y_pred[valid] - y_ref[valid]
    rmse = float(np.sqrt(np.mean(e ** 2)))

    if np.std(y_pred[valid]) < 1e-12 or np.std(y_ref[valid]) < 1e-12:
        return rmse, np.nan

    corr = float(np.corrcoef(y_pred[valid], y_ref[valid])[0, 1])
    return rmse, float(corr ** 2)


def draw_ik_compare(trial_key: str, side: str, unit: str, t_window: tuple[float, float]):
    d = TRIAL_DATA[trial_key]
    t = d['t'] - d['t'][0]
    imu_ik = d['imu_ik_rad']
    mocap_ik = d['mocap_ik_rad']

    if unit == 'deg':
        scale = 180.0 / np.pi
        ylab = 'Angle (deg)'
    else:
        scale = 1.0
        ylab = 'Angle (rad)'

    channels = IK_PLOT_CHANNELS[side]
    t0, t1 = t_window

    fig, axs = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

    metric_rows = []
    for i, (joint_name, ik_col) in enumerate(channels):
        ax = axs[i]
        idx = IK_DOF_NAMES.index(ik_col)

        y_imu = imu_ik[:, idx] * scale
        y_mocap = mocap_ik[:, idx] * scale

        rmse, r2 = compute_rmse_r2(y_imu, y_mocap)

        ax.plot(t, y_mocap, label='Mocap IK', color='#1e88e5', lw=1.8)
        ax.plot(t, y_imu, label='IMU IK (knee sign flipped)', color='#e53935', lw=1.4, ls='--')
        ax.set_title(f"{joint_name} {side} | RMSE={rmse:.3f} {unit}, R²={r2:.3f}")
        ax.axhline(0.0, color='gray', lw=0.7, ls=':')
        ax.set_xlim(t0, t1)
        ax.set_ylabel(ylab)
        ax.spines[['top', 'right']].set_visible(False)
        if i == 0:
            ax.legend(loc='upper right')

        metric_rows.append({'joint': f'{joint_name} {side}', 'rmse': rmse, 'r2': r2})

    axs[-1].set_xlabel('Time (s)')
    fig.suptitle(
        f"{trial_key} | IMU vs Mocap IK after sync ({d.get('align_method', 'unknown')}) | "
        f"lag={d.get('lag_samples', 0):+d} samples ({d.get('lag_seconds', 0.0):+.3f} s)",
        fontsize=12,
    )
    fig.tight_layout()
    plt.show()

    metric_df = pd.DataFrame(metric_rows)
    display(metric_df)
    print(
        f"Mean {side} RMSE={metric_df['rmse'].mean():.3f} {unit} | "
        f"Mean {side} R²={metric_df['r2'].mean():.3f}"
    )


def redraw_ik(*_):
    ik_out.clear_output(wait=True)
    with ik_out:
        draw_ik_compare(ik_trial_dd.value, ik_side_dd.value, ik_unit_dd.value, ik_time_range.value)


def on_ik_trial_change(change):
    if change.get('name') == 'value':
        set_ik_time_slider_for_trial(change['new'])
        redraw_ik()


ik_trial_dd.observe(on_ik_trial_change, names='value')
ik_side_dd.observe(redraw_ik, names='value')
ik_unit_dd.observe(redraw_ik, names='value')
ik_time_range.observe(redraw_ik, names='value')

set_ik_time_slider_for_trial(ik_trial_dd.value)
display(widgets.VBox([widgets.HBox([ik_trial_dd, ik_side_dd, ik_unit_dd]), ik_time_range]), ik_out)
redraw_ik()

Output()